# Step 10d: Train bge-base on PubMed Pairs

Replication of the arXiv fine-tuning experiment on a biomedical domain.

**Training data:** `pubmed_training_pairs.json` (1300 pairs, gemma3:4b generated)
**Validation:** `pubmed_manual_val.json` (10 human PubMed queries)
**Output:** `experiments/pubmed_bge/`

Everything else identical to all previous runs: bge-base-en-v1.5,
MNRL, 5 epochs, batch 32, warmup 100, seed 42.

## 0. Runtime check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())
if 'T4' not in result.stdout:
    print('WARNING: Not on T4. Go to Runtime > Change runtime type > T4 GPU')
else:
    print('Good to go.')

## 1. Install

In [ ]:
%%capture
!pip install sentence-transformers==5.5.1 faiss-cpu==1.13.2 accelerate

## 2. Upload files (run 3 times, one file each)

- `data/pubmed_training_pairs.json`
- `data/pubmed_manual_val.json`
- `experiments/pubmed_baseline/pubmed_baseline_chunks.pkl`

In [ ]:
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/experiments/pubmed_baseline', exist_ok=True)
os.makedirs('/content/experiments/pubmed_bge/model', exist_ok=True)

dest_map = {
    'pubmed_training_pairs.json':  '/content/data/pubmed_training_pairs.json',
    'pubmed_manual_val.json':      '/content/data/pubmed_manual_val.json',
    'pubmed_baseline_chunks.pkl':  '/content/experiments/pubmed_baseline/pubmed_baseline_chunks.pkl',
}

print('Upload one file at a time — run this cell 3 times')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = dest_map.get(fname, f'/content/{fname}')
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'✓ {fname} -> {dest}')

print('\nStatus:')
for path in dest_map.values():
    print(f'  {"✓" if os.path.exists(path) else "✗ missing"}  {path}')

## 3. Config

In [ ]:
import random, numpy as np

BASE_MODEL   = 'BAAI/bge-base-en-v1.5'
TRAIN_SIZE   = 1300
EPOCHS       = 5
BATCH_SIZE   = 32
WARMUP_STEPS = 100
SEED         = 42
OUT_DIR      = '/content/experiments/pubmed_bge/model'

random.seed(SEED)
np.random.seed(SEED)
print(f'Base model : {BASE_MODEL}')
print(f'Train size : {TRAIN_SIZE}')
print(f'Epochs     : {EPOCHS}')

## 4. Load data

In [ ]:
import json

with open('/content/data/pubmed_training_pairs.json') as f:
    all_pairs = json.load(f)
with open('/content/data/pubmed_manual_val.json') as f:
    val_pairs = json.load(f)

# Last 100 = held-out synthetic eval, never touch
train_pairs = all_pairs[:TRAIN_SIZE]

print(f'Total pairs      : {len(all_pairs)}')
print(f'Training pairs   : {len(train_pairs)}')
print(f'Val queries      : {len(val_pairs)} (human-written)')
print(f'Held-out eval    : {len(all_pairs) - TRAIN_SIZE - 100} val + 100 synthetic')

## 5. Validation evaluator (human PubMed queries)

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

queries  = {str(i): p['query']    for i, p in enumerate(val_pairs)}
corpus   = {str(i): p['positive'] for i, p in enumerate(val_pairs)}
relevant = {str(i): {str(i)}      for i in range(len(val_pairs))}

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant, name='val-pubmed-human')
print(f'Evaluator ready — {len(val_pairs)} human PubMed val queries')

## 6. Train (~10 min on T4)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

examples = [InputExample(texts=[p['query'], p['positive']]) for p in train_pairs]
model    = SentenceTransformer(BASE_MODEL)
loader   = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss     = MultipleNegativesRankingLoss(model)

print(f'Training — {len(examples)} pairs, {EPOCHS} epochs')

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=OUT_DIR,
    save_best_model=True,
    show_progress_bar=True,
    use_amp=True
)
print(f'\nSaved to {OUT_DIR}')

## 7. Build FAISS index

In [ ]:
import pickle, faiss

with open('/content/experiments/pubmed_baseline/pubmed_baseline_chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)
print(f'Loaded {len(chunks)} chunks')

model      = SentenceTransformer(OUT_DIR)
texts      = [c['text'] for c in chunks]
embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                           normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

idx_path = '/content/experiments/pubmed_bge/pubmed_bge.faiss'
faiss.write_index(index, idx_path)
print(f'Saved: {index.ntotal} vectors, dim={embeddings.shape[1]}')

## 8. Download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/pubmed_bge', 'zip', '/content/experiments', 'pubmed_bge')
files.download('/content/pubmed_bge.zip')
print('Unzip into your experiments/ folder on Mac.')
print('Then run: python step10e_eval_pubmed.py --eval manual')
print('      and: python step10e_eval_pubmed.py --eval synthetic')